# Building Dimensions table for the drivers silver tables

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
target_name = f"{catalog_name}.{gold_schema}.dim_drivers"
drivers_table = f"{catalog_name}.{silver_schema}.drivers"
nationality_table = f"{catalog_name}.{gold_schema}.ref_nationality_region"

### Read the silver tables

In [0]:
drivers_df = spark.table(drivers_table)
nationality_df = spark.table(nationality_table)

### Join the drivers and nationality tables and rename the region column

In [0]:
dim_drivers_df = (
    drivers_df.join(
        nationality_df, drivers_df.nationality == nationality_df.nationality, "left"
        ).select(drivers_df.driver_id, 
                 drivers_df.driver_name,
                 drivers_df.date_of_birth,
                 drivers_df.nationality, 
                 nationality_df.region.alias("nationality_region"))
)

In [0]:
display(dim_drivers_df)

### Write the dataframe into the gold schema

In [0]:
(
    dim_drivers_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(target_name)
)

In [0]:
display(spark.table(target_name))